<a href="https://colab.research.google.com/github/Samriddhacoderho/Face_Mask_Detection_CNN/blob/main/face_mask_detection_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torchvision import datasets, transforms

In [2]:
!pip install opendatasets
import opendatasets as od
od.download('https://www.kaggle.com/datasets/omkargurav/face-mask-dataset')

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: samriddharajsatyal
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/omkargurav/face-mask-dataset


100%|██████████| 163M/163M [00:04<00:00, 37.2MB/s]


In [3]:
dataset=datasets.ImageFolder('/content/face-mask-dataset/data')

In [4]:
dataset.class_to_idx

{'with_mask': 0, 'without_mask': 1}

In [5]:
#Transformations, Train-Test, DataSet
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, DataLoader

train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])


idx = np.arange(len(dataset))

train_idx, temp_idx = train_test_split(
    idx, test_size=0.2,
    stratify=dataset.targets,
    random_state=42
)

val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5,
    stratify=np.array(dataset.targets)[temp_idx],
    random_state=42
)

train_dataset = Subset(
    datasets.ImageFolder('/content/face-mask-dataset/data', transform=train_transform),
    train_idx
)

val_dataset = Subset(
    datasets.ImageFolder('/content/face-mask-dataset/data', transform=test_transform),
    val_idx
)

test_dataset = Subset(
    datasets.ImageFolder('/content/face-mask-dataset/data', transform=test_transform),
    test_idx
)

In [6]:
#Data Loaders
train_loader=DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader=DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader=DataLoader(test_dataset, batch_size=32, shuffle=False)

In [7]:
img,lbl=next(iter(train_loader))
img

tensor([[[[-0.2684, -0.2856, -0.3198,  ..., -0.2684, -0.2684, -0.2684],
          [-0.1657, -0.1657, -0.1828,  ..., -0.2513, -0.2684, -0.2684],
          [-0.2171, -0.2342, -0.2342,  ..., -0.2513, -0.2513, -0.2513],
          ...,
          [ 0.1254,  0.1939,  0.2624,  ...,  0.0569,  0.0227, -0.0116],
          [ 0.0912,  0.1426,  0.2282,  ...,  0.0569,  0.0398,  0.0056],
          [ 0.0398,  0.0741,  0.1426,  ...,  0.0741,  0.0398,  0.0056]],

         [[-0.1275, -0.1275, -0.1625,  ..., -0.1275, -0.1275, -0.1275],
          [-0.0049, -0.0224, -0.0399,  ..., -0.1099, -0.1275, -0.1275],
          [-0.0574, -0.0749, -0.0749,  ..., -0.1099, -0.1099, -0.1099],
          ...,
          [ 0.2052,  0.2752,  0.3627,  ...,  0.1176,  0.1001,  0.0651],
          [ 0.1702,  0.2227,  0.2927,  ...,  0.1176,  0.1176,  0.0826],
          [ 0.1176,  0.1527,  0.2227,  ...,  0.1352,  0.1176,  0.0826]],

         [[ 0.0256,  0.0256,  0.0082,  ..., -0.1661, -0.1661, -0.1661],
          [ 0.1476,  0.1302,  

In [8]:
#model creation

import torch.nn as nn
from torchvision import models

model=models.mobilenet_v2(weights='DEFAULT')

for param in model.parameters():
  param.requires_grad=False

num_ftrs=model.classifier[1].in_features

model.classifier=nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(num_ftrs, 128),
    nn.ReLU(),
    nn.Linear(128, 2) # 2 classes: Mask, No Mask
)



Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 75.1MB/s]


In [9]:
print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=128, bias=True)
  (2): ReLU()
  (3): Linear(in_features=128, out_features=2, bias=True)
)


In [10]:
print(model.classifier[1])

Linear(in_features=1280, out_features=128, bias=True)


In [11]:
import torch.optim as optim

criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.classifier.parameters(),lr=0.001)

In [12]:
from torch.autograd import no_grad
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=model.to(device)

#training loop
best_val_loss=float('inf')
patience=5
counter=0
min_delta=0.001
train_losses=[]
val_losses=[]
val_acc=[]
for epoch in range(10):
  tot_train_loss_per_epoch,tot_val_loss_per_epoch=0,0
  model.train()
  for img,lbl in train_loader:
    img=img.to(device)
    lbl=lbl.to(device)

    optimizer.zero_grad()
    y_pred=model(img)
    loss_train=criterion(y_pred,lbl)
    loss_train.backward()
    optimizer.step()
    tot_train_loss_per_epoch+=loss_train.item()
  train_losses.append(tot_train_loss_per_epoch/len(train_loader))

  model.eval()
  with torch.no_grad():
    tot,corr=0,0
    for img,lbl in val_loader:
      img=img.to(device)
      lbl=lbl.to(device)

      y_pred=model(img)
      loss_val=criterion(y_pred,lbl)
      tot_val_loss_per_epoch+=loss_val.item()

      _,preds=torch.max(y_pred,1)
      tot+=lbl.size(0)
      corr+=(preds==lbl).sum().item()

    val_losses.append(tot_val_loss_per_epoch/len(val_loader))
    val_acc.append(corr/tot)

  if((best_val_loss-(tot_val_loss_per_epoch/len(val_loader)))>=min_delta):
    best_val_loss=(tot_val_loss_per_epoch/len(val_loader))
    counter=0
    torch.save(model.state_dict(), 'mask_detector_model.pth')
  else:
    counter+=1
    if(counter>=patience):
      break
  print(f'Epoch {epoch+1}, Train Loss: {tot_train_loss_per_epoch/len(train_loader)}, Val Loss: {tot_val_loss_per_epoch/len(val_loader)}, Val Acc: {corr/tot}')



/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1, Train Loss: 0.18862148535945428, Val Loss: 0.11058095702901483, Val Acc: 0.9576158940397351
Epoch 2, Train Loss: 0.13681775274336652, Val Loss: 0.07188365492038429, Val Acc: 0.976158940397351
Epoch 3, Train Loss: 0.11747298664635136, Val Loss: 0.07291663302263866, Val Acc: 0.9708609271523179
Epoch 4, Train Loss: 0.11750735884049425, Val Loss: 0.06960031401831657, Val Acc: 0.9682119205298013
Epoch 5, Train Loss: 0.10885731677066475, Val Loss: 0.06772949430160224, Val Acc: 0.9735099337748344
Epoch 6, Train Loss: 0.10222907439514838, Val Loss: 0.06799305641713242, Val Acc: 0.9695364238410596
Epoch 7, Train Loss: 0.10355254505125303, Val Loss: 0.06262252627251048, Val Acc: 0.9774834437086093
Epoch 8, Train Loss: 0.08360762720208162, Val Loss: 0.057036278885789216, Val Acc: 0.9774834437086093
Epoch 9, Train Loss: 0.08742589505744122, Val Loss: 0.051927399957397334, Val Acc: 0.9788079470198675
Epoch 10, Train Loss: 0.08523845447414609, Val Loss: 0.05594316545951491, Val Acc: 0.97615